In [55]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on 2024-05-16

@author: Juan Enrique López Marcos

@description: Jupyter Notebook creado para obtener las técnicas a partir de los catálogos de reglas de detección (Sentinel, Splunk y QRadar) 

"""

'\nCreated on 2024-05-16\n\n@author: Juan Enrique López Marcos\n\n@description: Jupyter Notebook creado para obtener las técnicas a partir de los catálogos de reglas de detección (Sentinel, Splunk y QRadar) \n\n'

**Requerimientos**

In [56]:
import csv
import os
import requests
import pandas as pd
from stix2 import Filter, MemoryStore
import stix2
# ------------
from attackcti import attack_client
# ------------
from IPython.display import display
pd.options.display.max_columns = None

**Funciones y cliente**

In [57]:
# Deprecated
# def techniques(lift):
#     #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
#     techniques = lift.get_enterprise_techniques(stix_format=False)
#     techniques = pd.json_normalize(techniques)
#     #Eliminar las tecnicas deprecadas y revocadas
#     techniques = techniques[(techniques['mitre_deprecated'] != True)]
#     # Eliminamos duplicados y convertimos en lista
#     techniques = techniques['technique_id'].drop_duplicates().tolist()
#     return techniques


In [58]:
# get_data_from_branch () y  get_list_techniques_from_stix2() reemplazan a techniques()
# Acceso a los datos MITRE via request directa al json publicado en GitHub. Importante: debemos pasarle como parámetro la matriz
def get_data_from_branch(matrix):
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    stix_json = requests.get(url).json()
    return MemoryStore(stix_data=stix_json["objects"])

In [59]:
# Obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan técnicas y subtenicas pudiendose seleccionar
def get_list_techniques_from_stix2(src, include="both"):
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques


In [60]:
# Deprecated
# def techniques_info(lift):
#     techniques_info = lift.get_enterprise_techniques(stix_format=False)
#     techniques_info = pd.json_normalize(techniques_info)
#     #Eliminar las tecnicas deprecadas y revocadas
#     techniques_info = techniques_info[(techniques_info['mitre_deprecated'] != True)]
#     techniques_info = techniques_info[['technique_id','type', 'url', 'technique', 'technique_description','tactic','data_sources']]
#     return techniques_info

In [61]:
# Obtención de la tabla de técnicas a partir de objeto stix2. Por defecto se recogen técnicas y subtécnicas.
def get_techniques_with_info_from_stix2(src, include="both"):
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df['url'] = stix2_df['external_references'].apply(lambda refs: refs[0].url if refs else None)
    stix2_df.rename(columns={'name': 'technique', 'description': 'technique_description'}, inplace=True)

    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()

    stix2_df = stix2_df[['technique_id','technique', 'technique_description','type', 'url']]

    return stix2_df

In [62]:
# Buscamos en la columna de descripción
def find_techniques(texto, techniques_list):
    if isinstance(texto, str):
        found = []
        for item in techniques_list:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

In [63]:
# Función diseñada para obtener el listado de técnicas (ttp id) junto con la query a partir del fichero /UCM Catalog 2024 [Sentinel].xlsx
def get_techniques_from_UCM_Sentinel(ucm_sentinel_df, techniques_list, mitre_matrix):
    # Formateos básicos
    ucm_sentinel_df['techniques'] = ucm_sentinel_df['techniques'].str.replace('[','')
    ucm_sentinel_df['techniques'] = ucm_sentinel_df['techniques'].str.replace(']','')
    ucm_sentinel_df['techniques'] = ucm_sentinel_df['techniques'].str.replace('"','')
    ucm_sentinel_df['techniques'] = ucm_sentinel_df['techniques'].str.split(',')
    ucm_sentinel_df = ucm_sentinel_df.explode('techniques').reset_index()
    ucm_sentinel_df['QUERY'] = ucm_sentinel_df['query'].str.upper()
    ucm_sentinel_df['QUERY'] = ucm_sentinel_df['QUERY'].fillna('N/A')

    # En primer lugar construmos un df que contenga la ttp, la query y un campo que identifique donde se ha obtenido, en este primer caso directamente desde el campo informado
    ucm_sentinel_raw = ucm_sentinel_df.assign(type_source='ttp from Sentinel - techniques')[['techniques', 'query','type_source']]
    ucm_sentinel_raw = ucm_sentinel_raw[ucm_sentinel_raw['techniques']!='']
    ucm_sentinel_raw = ucm_sentinel_raw.dropna(subset=['techniques'])
    ucm_sentinel_raw = ucm_sentinel_raw.drop_duplicates()
    # Filtramos las ttp's por la matriz buscada
    ucm_sentinel_raw = ucm_sentinel_raw[ucm_sentinel_raw['techniques'].isin(techniques_list)]


    # Copiamos el df original y hacemos la búsqueda sobre el campo query
    ucm_sentinel_query = ucm_sentinel_df
    # Buscamos en el contenido menciones a la ttp
    ucm_sentinel_query['techniques_from_query'] = ucm_sentinel_query['query'].apply(lambda x: find_techniques(x, techniques_list))
    # Filtramos filas en las que no se haya encontrado nada
    ucm_sentinel_query = ucm_sentinel_query[ucm_sentinel_query['techniques_from_query']!='']
    ucm_sentinel_query = ucm_sentinel_query.dropna(subset=['techniques_from_query'])
    # Como se añade como lista debemos crear tantas filas como ttp hayamos encontrado
    ucm_sentinel_query['techniques_from_query_explode'] = ucm_sentinel_query['techniques_from_query'].str.split(',')
    ucm_sentinel_query = ucm_sentinel_query.explode('techniques_from_query_explode')
    ucm_sentinel_query = ucm_sentinel_query.drop_duplicates()
    ucm_sentinel_query = ucm_sentinel_query.reset_index(drop=True)
    # FIltramos finalmente el df con el que vamos a trabajar
    ucm_sentinel_query = ucm_sentinel_query.assign(type_source='ttp from Sentinel - description')[['techniques_from_query_explode','query', 'type_source']]
    ucm_sentinel_query.rename(columns={'techniques_from_query_explode': 'techniques'}, inplace=True)

    # Ultimo paso: unir los resultados obtenidos directamente en la columna informada con los obtenidos en la busqueda en query
    ucm_sentinel_union = pd.concat([ucm_sentinel_raw, ucm_sentinel_query])
    ucm_sentinel_union['techniques'] = ucm_sentinel_union['techniques'].str.strip()
    ucm_sentinel_union = ucm_sentinel_union.drop_duplicates()
    ucm_sentinel_union.rename(columns={'techniques': 'technique_id'}, inplace=True)
    ucm_sentinel_union = ucm_sentinel_union.sort_values(by='technique_id')

    print(f"Se han generado un total de {ucm_sentinel_union.shape[0]} filas para la matriz {mitre_matrix.upper()}. Las cuales se corresponden con {len(ucm_sentinel_union['technique_id'].unique())} ttp's únicas.")

    return ucm_sentinel_union


In [64]:
# Función diseñada para obtener el listado de técnicas (ttp id) junto con la query y su descripción a partir del fichero /UCM Catalog 2024 [Splunk].xlsx
# Splunk no dispone de campo informado como tal de técnica pero tiene un campo donde se almacena un diccionario en el que aparece el parametro mittre-attack y la ttp e caso de disponer de ella.
def get_techniques_from_UCM_Splunk(ucm_splunk_df, techniques_list, mitre_matrix):

    # Formateo básico
    ucm_splunk_df['MITRE_TECHNIQUE'] = ucm_splunk_df['Mitre Technique'].str.upper() # Ponemos el contenido en mayúsculas para encontrar coincidencias más adelante
    ucm_splunk_df['DESCRIPTION'] = ucm_splunk_df['description'].str.upper()
    ucm_splunk_df['SEARCH'] = ucm_splunk_df['search'].str.upper()


    # # Comenzamos generando el df con las técnicas encontradas en el campo 'MITRE_TECHNIQUE'
    # ucm_splunk_mitretechnique = ucm_splunk_df
    # ucm_splunk_mitretechnique['techniques_from_mt'] = ucm_splunk_mitretechnique['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    # ucm_splunk_mitretechnique = ucm_splunk_mitretechnique[ucm_splunk_mitretechnique['techniques_from_mt']!='']
    # ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.dropna(subset=['techniques_from_mt'])
    # ucm_splunk_mitretechnique['techniques_from_mt'] = ucm_splunk_mitretechnique['techniques_from_mt'].str.split(',')
    # ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.explode('techniques_from_mt')
    # ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.drop_duplicates().reset_index(drop=True)
    # ucm_splunk_mitretechnique.rename(columns={'techniques_from_mt': 'technique_id', 'search': 'query'}, inplace=True)
    # ucm_splunk_mitretechnique = ucm_splunk_mitretechnique[['technique_id','description','query']].sort_values(by='technique_id')


    # Comenzamos generando el df con las técnicas encontradas en el campo 'MITRE_TECHNIQUE'
    ucm_splunk_mitretechnique = ucm_splunk_df
    ucm_splunk_mitretechnique['techniques_from_mt'] = ucm_splunk_mitretechnique['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_splunk_mitretechnique = ucm_splunk_mitretechnique[ucm_splunk_mitretechnique['techniques_from_mt']!='']
    ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.dropna(subset=['techniques_from_mt'])
    ucm_splunk_mitretechnique['techniques_from_mt'] = ucm_splunk_mitretechnique['techniques_from_mt'].str.split(',')
    ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.explode('techniques_from_mt')
    ucm_splunk_mitretechnique = ucm_splunk_mitretechnique.drop_duplicates().reset_index(drop=True)
    ucm_splunk_mitretechnique.rename(columns={'techniques_from_mt': 'technique_id', 'search': 'query'}, inplace=True)
    ucm_splunk_mitretechnique = ucm_splunk_mitretechnique[['technique_id','description','query']].sort_values(by='technique_id')
    ucm_splunk_mitretechnique['type_source'] = 'ttp from Splunk - Mitre Technique'

    #Repetimos proceso para el campo description
    ucm_splunk_desc = ucm_splunk_df
    ucm_splunk_desc['techniques_from_description'] = ucm_splunk_desc['DESCRIPTION'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_splunk_desc = ucm_splunk_desc[ucm_splunk_desc['techniques_from_description']!='']
    ucm_splunk_desc = ucm_splunk_desc.dropna(subset=['techniques_from_description'])
    ucm_splunk_desc['techniques_from_search'] = ucm_splunk_desc['techniques_from_description'].str.split(',')
    ucm_splunk_desc = ucm_splunk_desc.explode('techniques_from_description')
    ucm_splunk_desc = ucm_splunk_desc.drop_duplicates().reset_index(drop=True)
    ucm_splunk_desc.rename(columns={'techniques_from_description': 'technique_id', 'search': 'query'}, inplace=True)
    ucm_splunk_desc = ucm_splunk_desc[['technique_id','description','query']].sort_values(by='technique_id')
    ucm_splunk_desc['type_source'] = 'ttp from Splunk - description'

    # Buscamos en el campo search
    ucm_splunk_search = ucm_splunk_df
    ucm_splunk_search['techniques_from_search'] = ucm_splunk_search['SEARCH'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_splunk_search = ucm_splunk_search[ucm_splunk_search['techniques_from_search']!='']
    ucm_splunk_search = ucm_splunk_search.dropna(subset=['techniques_from_search'])
    ucm_splunk_search['techniques_from_search'] = ucm_splunk_search['techniques_from_search'].str.split(',')
    ucm_splunk_search = ucm_splunk_search.explode('techniques_from_search')
    ucm_splunk_search = ucm_splunk_search.drop_duplicates().reset_index(drop=True)
    ucm_splunk_search.rename(columns={'techniques_from_search': 'technique_id', 'search': 'query'}, inplace=True)
    ucm_splunk_search = ucm_splunk_search[['technique_id','description','query']].sort_values(by='technique_id')
    ucm_splunk_search['type_source'] = 'ttp from Splunk - search'


    #Unimos finalmente los resultados
    ucm_splunk_union = pd.concat([ucm_splunk_mitretechnique, ucm_splunk_desc, ucm_splunk_search])

    print(f"Se han generado un total de {ucm_splunk_union.shape[0]} filas para la matriz {mitre_matrix.upper()}. Las cuales se corresponden con {len(ucm_splunk_union['technique_id'].unique())} ttp's únicas.")

    return ucm_splunk_union


In [65]:
# En el caso de Qradar disponemos de 6 dataframes ya que el libro xlsx está compuesto por 6 hojas distintas de las que obtendremos la información. 
def get_techniques_from_UCM_Qradar(ucm_qradar_baseline_df, ucm_qradar_ws_df, ucm_qradar_atomics_df, ucm_qradar_linux_df, ucm_qradar_fw_df, ucm_qradar_aws_df, techniques_list, mitre_matrix):

    # Primera hoja del .xlsx: Baseline
    # Baseline - Formateo básico
    ucm_qradar_baseline_df['MITRE_TECHNIQUE'] = ucm_qradar_baseline_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_baseline_mt = ucm_qradar_baseline_df

    ucm_qradar_baseline_mt['techniques_from_mt'] = ucm_qradar_baseline_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_baseline_mt = ucm_qradar_baseline_mt[ucm_qradar_baseline_mt['techniques_from_mt']!='']
    ucm_qradar_baseline_mt = ucm_qradar_baseline_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_baseline_mt['techniques_from_mt'] = ucm_qradar_baseline_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_baseline_mt = ucm_qradar_baseline_mt.explode('techniques_from_mt')
    ucm_qradar_baseline_mt = ucm_qradar_baseline_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_baseline_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_baseline_mt = ucm_qradar_baseline_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_baseline_mt['type_source'] = 'ttp from Qradar - Baseline - Mitre Technique'

    # Siguiente hoja: Windows|Sysmon
    # Windows|Sysmon - Formateo básico
    ucm_qradar_ws_df['MITRE_TECHNIQUE'] = ucm_qradar_ws_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_ws_mt = ucm_qradar_ws_df

    ucm_qradar_ws_mt['techniques_from_mt'] = ucm_qradar_ws_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_ws_mt = ucm_qradar_ws_mt[ucm_qradar_ws_mt['techniques_from_mt']!='']
    ucm_qradar_ws_mt = ucm_qradar_ws_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_ws_mt['techniques_from_mt'] = ucm_qradar_ws_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_ws_mt = ucm_qradar_ws_mt.explode('techniques_from_mt')
    ucm_qradar_ws_mt = ucm_qradar_ws_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_ws_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_ws_mt = ucm_qradar_ws_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_ws_mt['type_source'] = 'ttp from Qradar - Windows|Sysmon - Mitre Technique'


    # Siguiente hoja: Atomics
    # Atomics - Formateo básico
    ucm_qradar_atomics_df['MITRE_TECHNIQUE'] = ucm_qradar_atomics_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_atomics_mt = ucm_qradar_atomics_df

    ucm_qradar_atomics_mt['techniques_from_mt'] = ucm_qradar_atomics_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_atomics_mt = ucm_qradar_atomics_mt[ucm_qradar_atomics_mt['techniques_from_mt']!='']
    ucm_qradar_atomics_mt = ucm_qradar_atomics_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_atomics_mt['techniques_from_mt'] = ucm_qradar_atomics_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_atomics_mt = ucm_qradar_atomics_mt.explode('techniques_from_mt')
    ucm_qradar_atomics_mt = ucm_qradar_atomics_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_atomics_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_atomics_mt = ucm_qradar_atomics_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_atomics_mt['type_source'] = 'ttp from Qradar - Atomics - Mitre Technique'


    # Siguiente hoja: Linux
    # Linux - Formateo básico
    ucm_qradar_linux_df['MITRE_TECHNIQUE'] = ucm_qradar_linux_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_linux_mt = ucm_qradar_linux_df

    ucm_qradar_linux_mt['techniques_from_mt'] = ucm_qradar_linux_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_linux_mt = ucm_qradar_linux_mt[ucm_qradar_linux_mt['techniques_from_mt']!='']
    ucm_qradar_linux_mt = ucm_qradar_linux_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_linux_mt['techniques_from_mt'] = ucm_qradar_linux_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_linux_mt = ucm_qradar_linux_mt.explode('techniques_from_mt')
    ucm_qradar_linux_mt = ucm_qradar_linux_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_linux_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_linux_mt = ucm_qradar_linux_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_linux_mt['type_source'] = 'ttp from Qradar - Linux - Mitre Technique'

    # Siguiente hoja: FW
    # Linux - Formateo básico
    ucm_qradar_fw_df['MITRE_TECHNIQUE'] = ucm_qradar_fw_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_fw_mt = ucm_qradar_fw_df

    ucm_qradar_fw_mt['techniques_from_mt'] = ucm_qradar_fw_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_fw_mt = ucm_qradar_fw_mt[ucm_qradar_fw_mt['techniques_from_mt']!='']
    ucm_qradar_fw_mt = ucm_qradar_fw_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_fw_mt['techniques_from_mt'] = ucm_qradar_fw_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_fw_mt = ucm_qradar_fw_mt.explode('techniques_from_mt')
    ucm_qradar_fw_mt = ucm_qradar_fw_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_fw_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_fw_mt = ucm_qradar_fw_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_fw_mt['type_source'] = 'ttp from Qradar - Linux - Mitre Technique'


    # Siguiente hoja: AWS
    # Linux - Formateo básico
    ucm_qradar_aws_df['MITRE_TECHNIQUE'] = ucm_qradar_aws_df['Mitre Technique'].str.upper()
    # En este caso el campo en el que buscaremos siempre es 'Mitre Technique'
    ucm_qradar_aws_mt = ucm_qradar_aws_df

    ucm_qradar_aws_mt['techniques_from_mt'] = ucm_qradar_aws_mt['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x, techniques_list))
    ucm_qradar_aws_mt = ucm_qradar_aws_mt[ucm_qradar_aws_mt['techniques_from_mt']!='']
    ucm_qradar_aws_mt = ucm_qradar_aws_mt.dropna(subset=['techniques_from_mt'])
    ucm_qradar_aws_mt['techniques_from_mt'] = ucm_qradar_aws_mt['techniques_from_mt'].str.split(',')
    ucm_qradar_aws_mt = ucm_qradar_aws_mt.explode('techniques_from_mt')
    ucm_qradar_aws_mt = ucm_qradar_aws_mt.drop_duplicates().reset_index(drop=True)
    ucm_qradar_aws_mt.rename(columns={'techniques_from_mt': 'technique_id', 'Rule': 'rule'}, inplace=True)
    ucm_qradar_aws_mt = ucm_qradar_aws_mt[['technique_id','rule']].sort_values(by='technique_id')
    ucm_qradar_aws_mt['type_source'] = 'ttp from Qradar - AWS - Mitre Technique'

    # Unimos las tablas generadas
    ucm_qradar_union = pd.concat([ucm_qradar_baseline_mt, ucm_qradar_ws_mt, ucm_qradar_atomics_mt, ucm_qradar_linux_mt, ucm_qradar_fw_mt, ucm_qradar_aws_mt])
    ucm_qradar_union = ucm_qradar_union.drop_duplicates(subset=['technique_id', 'rule']) # Eliminamos duplicados sin tener en cuenta la plataforma
    ucm_qradar_union = ucm_qradar_union.sort_values(by='technique_id')

    print(f"Qradar: Se han generado un total de {ucm_qradar_union.shape[0]} filas para la matriz {mitre_matrix.upper()}. Las cuales se corresponden con {len(ucm_qradar_union['technique_id'].unique())} ttp's únicas.")

    return ucm_qradar_union


**Parámetros**

In [66]:
matrix = 'enterprise' # enterprise / ics / mobile

In [67]:
# Obtenemos el listado de técnicas a utilizar filtrado por matriz
data_raw = get_data_from_branch(matrix)
techniques = get_list_techniques_from_stix2(data_raw, 'both')
techniques[:5]

['T1630.001', 'T1430.002', 'T1655.001', 'T1521.002', 'T1636.002']

**Inputs**

In [68]:
# path_UCMCatalog2024 = r'C:\Users\jelopez\Documents\CyberProof\python\check_0905\UCM Catalog 2024' OLD
path_UCMCatalog2024 = os.path.join(os.getcwd(),'inputs', 'UCM Catalog 2024')

path_UCMCatalog2024_Sentinel = path_UCMCatalog2024 + '/UCM Catalog 2024 [Sentinel].xlsx'
path_UCMCatalog2024_Qradar = path_UCMCatalog2024 + '/UCM Catalog 2024 [Qradar].xlsx'
path_UCMCatalog2024_Splunk = path_UCMCatalog2024 + '/UCM Catalog 2024 [Splunk].xlsx'

In [69]:
# Cargamos la información en un dataframe. En el caso de Qradar tenemos que revisar 4 hojas distintas del libro
UCMCatalog2024_Sentinel = pd.read_excel(path_UCMCatalog2024_Sentinel)

UCMCatalog2024_Splunk = pd.read_excel(path_UCMCatalog2024_Splunk)

UCMCatalog2024_Qradar_baseline = pd.read_excel(path_UCMCatalog2024_Qradar,'Baseline')
UCMCatalog2024_Qradar_ws= pd.read_excel(path_UCMCatalog2024_Qradar,'Windows|Sysmon')
UCMCatalog2024_Qradar_linux= pd.read_excel(path_UCMCatalog2024_Qradar,'Linux')
UCMCatalog2024_Qradar_atomics = pd.read_excel(path_UCMCatalog2024_Qradar,'Atomics')
UCMCatalog2024_Qradar_fw= pd.read_excel(path_UCMCatalog2024_Qradar,'FW')
UCMCatalog2024_Qradar_aws= pd.read_excel(path_UCMCatalog2024_Qradar,'AWS')


#### **1. UCM Catalog 2024 - SENTINEL**

**Ejecución**

In [70]:
ucm_sentinel_union = get_techniques_from_UCM_Sentinel(UCMCatalog2024_Sentinel, techniques, matrix)
ucm_sentinel_union.head(3)

Se han generado un total de 22 filas para la matriz MOBILE. Las cuales se corresponden con 15 ttp's únicas.


,technique_id,query,type_source
919,T1420,CLCFortinet_CL\n| where isnotempty(Message)\n|...,ttp from Sentinel - techniques
902,T1420,let IoCList = externaldata(TimeGenerated:datet...,ttp from Sentinel - techniques
2266,T1421,AWSCloudWatchCustom_CL\n| extend parsed = tody...,ttp from Sentinel - techniques


**Guardado**

In [71]:
if not os.path.exists(os.path.join(os.getcwd(), 'outputs', matrix)):
    os.makedirs(os.path.join(os.getcwd(), 'outputs', matrix))
    
path_UCMCatalog2024_Sentinel_union = os.path.join(os.getcwd(), 'outputs', matrix, f'[MITRE-{matrix}]_techniques_UCMCatalog2024_Sentinel_withquery.csv') # Hay que replicarlo para cada guardado
ucm_sentinel_union.to_csv(path_UCMCatalog2024_Sentinel_union, sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)

#### **2. UCM Catalog 2024 - SPLUNK**

**Ejecución**

In [72]:
ucm_splunk_union = get_techniques_from_UCM_Splunk(UCMCatalog2024_Splunk, techniques, matrix)
ucm_splunk_union.head(3)

Se han generado un total de 0 filas para la matriz MOBILE. Las cuales se corresponden con 0 ttp's únicas.


,technique_id,description,query,type_source


**Guardado**

In [73]:
if not os.path.exists(os.path.join(os.getcwd(), 'outputs', matrix)):
    os.makedirs(os.path.join(os.getcwd(), 'outputs', matrix))
    
path_UCMCatalog2024_Splunk_union = os.path.join(os.getcwd(), 'outputs', matrix, f'[MITRE-{matrix}]_techniques_UCMCatalog2024_Splunk_withquery.csv') # Hay que replicarlo para cada guardado
ucm_splunk_union.to_csv(path_UCMCatalog2024_Splunk_union, sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)

#### **3. UCM Catalog 2024 - QRADAR**

**Ejecución**

In [74]:
ucm_qradar_union  = get_techniques_from_UCM_Qradar(UCMCatalog2024_Qradar_baseline, UCMCatalog2024_Qradar_ws, UCMCatalog2024_Qradar_atomics, UCMCatalog2024_Qradar_linux, UCMCatalog2024_Qradar_fw, UCMCatalog2024_Qradar_aws, techniques, matrix)
ucm_qradar_union.head(3)

Qradar: Se han generado un total de 0 filas para la matriz MOBILE. Las cuales se corresponden con 0 ttp's únicas.


,technique_id,rule,type_source


**Guardado**

In [75]:
if not os.path.exists(os.path.join(os.getcwd(), 'outputs', matrix)):
    os.makedirs(os.path.join(os.getcwd(), 'outputs', matrix))
    
path_UCMCatalog2024_Qradar_union = os.path.join(os.getcwd(), 'outputs', matrix, f'[MITRE-{matrix}]_techniques_UCMCatalog2024_Qradar_withquery.csv') # Hay que replicarlo para cada guardado
ucm_qradar_union.to_csv(path_UCMCatalog2024_Qradar_union, sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)

#### **4. Unión final de Sentinel, Splunk y Qradar**

In [76]:
ucm_sentinel_union.columns

Index(['technique_id', 'query', 'type_source'], dtype='object')

In [77]:
ucm_splunk_union.columns

Index(['technique_id', 'description', 'query', 'type_source'], dtype='object')

In [78]:
ucm_qradar_union.columns

Index(['technique_id', 'rule', 'type_source'], dtype='object')

In [79]:
ucm_sentinel_union['source'] = 'Sentinel'
ucm_splunk_union['source'] = 'Splunk'
ucm_qradar_union['source'] = 'Qradar'

In [80]:
UCMCatalog2024 = pd.concat([ucm_sentinel_union, ucm_splunk_union, ucm_qradar_union])
UCMCatalog2024 = UCMCatalog2024.sort_values(by='technique_id')
print(f"Se han generado un total de {UCMCatalog2024.shape[0]} filas para la matriz {matrix.upper()}. Las cuales se corresponden con {len(UCMCatalog2024['technique_id'].unique())} ttp's únicas.")


Se han generado un total de 22 filas para la matriz MOBILE. Las cuales se corresponden con 15 ttp's únicas.


In [81]:
if not os.path.exists(os.path.join(os.getcwd(), 'outputs', matrix)):
    os.makedirs(os.path.join(os.getcwd(), 'outputs', matrix))
    
path_UCMCatalog2024_union = os.path.join(os.getcwd(), 'outputs', matrix, f'[MITRE-{matrix}]_techniques_UCMCatalog2024_withquery.csv') # Hay que replicarlo para cada guardado
UCMCatalog2024.to_csv(path_UCMCatalog2024_union, sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)